In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(r'D:\DSAI\Project\data\人工评分.csv', encoding='gbk')[['text','score']].dropna()
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() >= 2]

train, test = train_test_split(df, test_size=0.1, random_state=42, stratify=df['score'].round(1))
train, dev   = train_test_split(train, test_size=0.1, random_state=42, stratify=train['score'].round(1))

for d,name in zip([train,dev,test],['train','dev','test']):
    d.to_csv(f'D:/DSAI/Project/data/{name}.csv', index=False, encoding='utf-8-sig')
print('done')

done


In [30]:
# ---------- STEP 1 数据 + 管道 ----------
import os, tensorflow as tf, pandas as pd, numpy as np
from transformers import BertTokenizerFast
from sklearn.model_selection import train_test_split

train_df = pd.read_csv('D:/DSAI/Project/data/train.csv')[['text','score']].dropna()
train_df['text'] = train_df['text'].astype(str).str.strip()
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df.score.round(1))

MAX_LEN, BATCH = 16, 1024
tokenizer = BertTokenizerFast.from_pretrained('bert-base-chinese')

def make_ds(df):
    enc = tokenizer(df.text.tolist(), truncation=True, padding='max_length', max_length=MAX_LEN)
    return tf.data.Dataset.from_tensor_slices((
        dict(input_ids=enc['input_ids'], attention_mask=enc['attention_mask']),
        df.score.astype('float32')
    )).batch(BATCH)

train_ds = make_ds(train_df).shuffle(1000)
dev_ds   = make_ds(dev_df)

C:\Users\86180\.conda\envs\DSAI\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [31]:
# ---------- STEP 2 模型：回归头（无激活） ----------
from transformers import TFBertForSequenceClassification

model = TFBertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=1)  # 默认 linear
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.02),
    loss='mse',                      # ← 关键：连续标签回归
    metrics=['mae']
)
model.summary()

C:\Users\86180\.conda\envs\DSAI\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification_14"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  102267648 
                                                                 
 dropout_569 (Dropout)       multiple                  0         
                                                                 
 classifier (Dense)          multiple                  769       
                                                                 
Total params: 102,268,417
Trainable params: 102,268,417
Non-trainable params: 0
_________________________________________________________________


In [ ]:
# ---------- STEP 3 训练 ----------
history = model.fit(
    train_ds,
    validation_data=dev_ds,
    epochs=1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
    ],
    verbose=2
)

In [15]:
SAVE_DIR = 'D:/DSAI/Project/best_model_tf'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('saved to', SAVE_DIR)

saved to D:/DSAI/Project/best_model_tf


In [ ]:
import os, tensorflow as tf, pandas as pd, numpy as np, glob, re
from transformers import BertTokenizerFast, TFBertForSequenceClassification
from tqdm import tqdm   # 仅新增这一行

# ------------ 提速三件套 ------------
MAX_LEN = 32
BATCH   = 512
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.keras.mixed_precision.set_global_policy('mixed_float16')

SAVE_DIR = r'D:/DSAI/Project/best_model_tf'
tokenizer = BertTokenizerFast.from_pretrained(SAVE_DIR)
model     = TFBertForSequenceClassification.from_pretrained(SAVE_DIR)

@tf.function(jit_compile=True, reduce_retracing=True)
def predict_batch(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='tf')
    return model(enc)[0][:, 0]

# ------------ 手动输入范围 ------------
start = 1
end   = 349

# ------------ 顺序跑 + 进度条 ------------
for period in tqdm(range(start, end + 1), desc='期数'):
    rows = []
    for txt in glob.glob(rf'D:\DSAI\Project\data\每期弹幕\第{period}期\*.txt'):
        cid = re.search(r'(\d+)\.txt$', txt).group(1)
        with open(txt, encoding='utf-8') as f:
            rows += [{'期数': period, 'cid': cid, 'text': line.strip()} for line in f if line.strip()]
    if not rows:
        tqdm.write(f'⚠  第 {period} 期无弹幕，跳过')
        continue

    df = pd.DataFrame(rows)
    enc = tokenizer(df.text.tolist(), truncation=True, padding=True, max_length=MAX_LEN)
    ds = tf.data.Dataset.from_tensor_slices(
        dict(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
    ).batch(BATCH)
    df['score'] = model.predict(ds, verbose=0).squeeze()
    out = f'D:/DSAI/Project/data/danmu_score_p{period}.csv'
    df[['期数', 'cid', 'text', 'score']].to_csv(out, index=False, encoding='utf-8-sig')
    tqdm.write(f'✅ 第 {period} 期完成 → {out}')

In [25]:
import tensorflow as tf, pandas as pd, time
from transformers import BertTokenizerFast, TFBertForSequenceClassification

# ---------- 模型一次加载 ----------
SAVE_DIR = r'D:/DSAI/Project/best_model_tf'
tokenizer = BertTokenizerFast.from_pretrained(SAVE_DIR)
model     = TFBertForSequenceClassification.from_pretrained(SAVE_DIR)

MAX_LEN = 32          # 极限截断
BATCH   = 512         # 大 batch

@tf.function(jit_compile=True)
def predict_batch(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='tf')
    return model(enc)[0][:, 0]

# ---------- 只跑 1 个 txt ----------
txt_path = r'D:/DSAI/Project/data/每期弹幕/第1期/第1期第1个视频82142406.txt'   # ← 改这里
with open(txt_path, encoding='utf-8') as f:
    texts = [line.strip() for line in f if line.strip()]

t0 = time.time()
scores = predict_batch(texts).numpy()
t1 = time.time()

# ---------- 秒级结果 ----------
df = pd.DataFrame({'text': texts, 'score': scores})
print(f'✅ 完成！共 {len(texts)} 条，耗时 {t1-t0:.2f} 秒')
print(df)          # 只看前 10 条

Some layers from the model checkpoint at D:/DSAI/Project/best_model_tf were not used when initializing TFBertForSequenceClassification: ['dropout_151']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at D:/DSAI/Project/best_model_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


✅ 完成！共 1000 条，耗时 97.27 秒
                 text     score
0                沾沾喜气  0.559082
1  天花板都不同，说努力和坚持的真搞笑！  0.558594
2     龚玉林我们也要永远在一起！！！  0.559082
3            龚玉林我永远爱你  0.558105
4            哦哦哦这天我生日  0.559082
5           一辈子再次触动了我  0.558594
6         !这是我见过最美的爱情  0.558594
7                 太棒了  0.559082
8              马国凤我爱你  0.558105
9        !我现在就处于中英异国恋  0.558105


In [26]:
df

,text,score
0,沾沾喜气,0.559082
1,天花板都不同，说努力和坚持的真搞笑！,0.558594
2,龚玉林我们也要永远在一起！！！,0.559082
3,龚玉林我永远爱你,0.558105
4,哦哦哦这天我生日,0.559082
...,...,...
995,祝福,0.558594
996,好可爱哈哈哈哈哈哈哈哈哈哈哈哈,0.561035
997,槜,0.559082
998,再见，王君。,0.558594


In [27]:
train_df.score.describe()

count    5537.000000
mean        0.547679
std         0.251320
min         0.000000
25%         0.500000
50%         0.500000
75%         0.750000
max         1.000000
Name: score, dtype: float64

In [ ]:
import pandas as np, pandas as pd

# 1. 读视频信息
info = pd.read_csv(r'D:\DSAI\Project\data\每周必看视频信息.csv', usecols=['number','cid','danmaku','pid_name_v2'])
info['cid'] = info['cid'].astype(str)

# 2. 读弹幕分数
danmu = pd.read_csv('D:/DSAI/Project/data/all_danmu_score_tf.csv')
danmu['cid'] = danmu['cid'].astype(str)

# 3. 合并
tmp = info[['number','cid','danmaku','pid_name_v2']].rename(columns={'danmaku':'weight'})
danmu = danmu.merge(tmp, on='cid', how='left')

# 4. 分区基线
tag_base = danmu.groupby(['number','cid']).apply(
    lambda g: np.average(g['score'], weights=g['weight'])).reset_index(name='video_raw_score')
tag_base = tag_base.merge(info[['cid','pid_name_v2']], on='cid')
baseline = tag_base.groupby('pid_name_v2').apply(
    lambda g: np.average(g['video_raw_score'], weights=g['weight'])).reset_index(name='tag_baseline')

# 5. 校正
tag_base = tag_base.merge(baseline, on='pid_name_v2', how='left')
tag_base['adj_score'] = (tag_base['video_raw_score'] - tag_base['tag_baseline'] + 0.5).clip(0,1)

# 6. 周聚合
raw_week  = tag_base.groupby('number').apply(lambda g: np.average(g['video_raw_score'], weights=g['weight']))
adj_week  = tag_base.groupby('number').apply(lambda g: np.average(g['adj_score'], weights=g['weight']))

out = pd.DataFrame({'number': raw_week.index, 'raw': raw_week.values, 'adj_tag': adj_week.values})
out.to_csv('D:/DSAI/Project/data/weekly_sentiment_tf.csv', index=False)
print('saved weekly_sentiment_tf.csv')

In [ ]:
# 可选：加载人工测试集，计算 Pearson
import pandas as pd
from scipy.stats import pearsonr

test_df = pd.read_csv('D:/DSAI/Project/data/dev.csv')
pred = predict_batch(test_df.text.tolist())
print('TensorFlow 版验证集 Pearson:', pearsonr(pred, test_df.score)[0])

In [6]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--user", "--force-reinstall",
    "transformers<4.30", "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"
])

0